# MAPP 3a: лексикалық түзеткіш (CSE + морфология) — Whisper large-v3 шығысында

Бірыңғай хаттамамен салыстыру үшін large-v3 RAW-ында лексикалық нұсқаны қайта жүргізеді (детерминді код, бұрынғы `fixed` бағанының орнына).
Кіріс: `dataset_wav16k/mapp_v2_batch_results_asr_upgrade.csv` (large-v3 + chunk). Шығыс: `dataset_wav16k/largev3_lex/mapp_v2_batch_results_largev3_lex.csv`.
Келесі дәптер — `MAPP_3b_largev3_MaxEnt_CRF.ipynb`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install rapidfuzz openpyxl xlrd==2.0.1

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.3 MB/s eta 0:00:00


In [2]:
import os, glob
D = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k"
def first_existing(*paths):
    for p in paths:
        if os.path.exists(p): return p
    raise FileNotFoundError("Табылмады: " + " | ".join(paths))

STEMS_PATH   = first_existing(D+"/qaz_stems_unik_edit.xlsx", "/content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH/qaz_stems_unik_edit.xlsx")
ENDINGS_PATH = first_existing(D+"/qaz_endings_morph_5994.xls", D+"/qaz_ending_5994.xls", "/content/drive/MyDrive/Colab Notebooks/MAPP_ASR/KAZAKH/qaz_ending_5994.xls")
STOP_PATH    = D+"/stop_words.txt"
print(STEMS_PATH, ENDINGS_PATH, STOP_PATH, os.path.exists(STOP_PATH), sep="\n")
if not os.path.exists(STOP_PATH):
    print("[ЕСКЕРТУ] stop_words.txt табылмады — бұрынғы жүгіруден өзгеше нәтиже беруі мүмкін. Файлды жоғарыдағы жолға салыңыз.")

/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/qaz_stems_unik_edit.xlsx
/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/qaz_endings_morph_5994.xls
/content/drive/MyDrive/Colab Notebooks/dataset_wav16k/stop_words.txt
True


In [3]:
import os
import re, math, itertools
from dataclasses import dataclass
from typing import List, Tuple
from collections import Counter
import pandas as pd
from rapidfuzz import fuzz, process

# ---- сөздіктерді жүктеу (xlsx/xls екеуі де болады; 1-қатар = тақырып) ----
def _read_table(path):
    return pd.read_excel(path, header=None)

def load_resources(stems_path, endings_path, stop_path=None):
    global stems_list, stems_set, endings_list, endings_md_list, ending_to_md, endings_set, stop_words
    global stems_lower, stem_freq
    st = _read_table(stems_path)
    stems = [str(x).replace("\ufeff", "").strip() for x in st.iloc[:, 0].dropna()]
    stems_list = sorted(set(s for s in stems if s), key=lambda s: (-len(s), s))   # детерминді реттеу
    en = _read_table(endings_path).iloc[1:]
    endings_list = [str(x).replace("\ufeff", "").strip() for x in en.iloc[:, 0]]
    endings_md_list = [str(x).replace("\ufeff", "").strip() if pd.notna(x) else "" for x in en.iloc[:, 1]]
    stop_words = set()
    if stop_path and os.path.exists(stop_path):
        stop_words = {l.strip().lower() for l in open(stop_path, encoding="utf-8") if l.strip()}
    stems_set = set(s.lower() for s in stems_list)
    ending_to_md = {endings_list[i]: endings_md_list[i] for i in range(len(endings_list))}
    endings_set = set(endings_list)
    stems_lower = [s.lower() for s in stems_list]
    stem_freq = Counter(s.lower() for s in stems_list)
    print("stems:", len(stems_list), "endings:", len(endings_list), "stopwords:", len(stop_words))


WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize_keep_punct(text: str) -> List[str]:
    return WORD_RE.findall(text)

def is_word(token: str) -> bool:
    return token.isalpha()

def normalize(token: str) -> str:
    # Важно: не ломаем казахские буквы. Просто lower().
    return token.lower()

def levenshtein(a: str, b: str, max_dist: int = 2) -> int:
    """
    Быстрый Левенштейн с отсечкой: если > max_dist — возвращаем max_dist+1
    """
    if a == b:
        return 0
    if abs(len(a) - len(b)) > max_dist:
        return max_dist + 1
    # DP по строке
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        cur = [i]
        min_row = cur[0]
        for j, cb in enumerate(b, start=1):
            ins = cur[j-1] + 1
            dele = prev[j] + 1
            sub = prev[j-1] + (ca != cb)
            v = min(ins, dele, sub)
            cur.append(v)
            if v < min_row:
                min_row = v
        prev = cur
        if min_row > max_dist:
            return max_dist + 1
    return prev[-1]


@dataclass
class Analysis:
    ok: bool
    stem: str
    ending: str
    md: str

def split_stem_ending(word: str, min_stem_len: int = 2) -> Tuple[str, str]:
    w = word
    wl = len(w)
    if wl <= min_stem_len:
        return w, ""

    wlow = w.lower()
    if wlow in stems_set:
        return w, ""

    # перебор возможных окончаний: от коротких к длинным/или наоборот
    # у вас: i от n+1 вниз (т.е. пробуем разные длины ending)
    # сделаем: пробуем ending от 1..(wl-min_stem_len), но эффективнее — от длинного к короткому
    max_ending_len = wl - min_stem_len
    for elen in range(max_ending_len, 0, -1):
        ending = w[wl-elen:]
        stem = w[:wl-elen]
        if ending in endings_set and stem.lower() in stems_set:
            return stem, ending

    # если не нашли, возвращаем "как есть"
    return w, ""

def cse_analyze_word(word: str) -> Analysis:
    stem, ending = split_stem_ending(word)
    if ending:
        md = ending_to_md.get(ending, "")
        return Analysis(ok=True, stem=stem, ending=ending, md=md)
    # ok=True если хотя бы stem — известный
    if stem.lower() in stems_set:
        return Analysis(ok=True, stem=stem, ending="", md="")
    return Analysis(ok=False, stem=stem, ending="", md="")


PHONETIC_PAIRS = [
    ("қ","к"), ("ғ","г"),
    ("ұ","у"), ("ү","у"),
    ("ы","і"), ("о","ө")
]
FUNCTION_WORDS = {
    "бірақ","және","немесе","ал","өйткені","сондықтан","егер","онда",
    "мен","сен","ол","біз","сіз","олар",
    "да","де","та","те",
    "емес","екен","еді"
}
ASR_NORMALIZE_MAP = {
    "биз": "біз",
    "бирак": "бірақ",
    "казир": "қазір",
    "себеби": "себебі",
    "жане": "және",
    "озимди": "өзімді",
    "оз": "өз",
    "бугин": "бүгін",
    "озимди": "өзімді",
}


def normalize_asr_token(w: str) -> str:
    wl = w.lower()
    return ASR_NORMALIZE_MAP.get(wl, wl)

def phonetic_variants(word: str) -> List[str]:
    cands = set()
    for a, b in PHONETIC_PAIRS:
        if a in word:
            cands.add(word.replace(a, b))
        if b in word:
            cands.add(word.replace(b, a))
    return list(cands)

def repair_by_endings(word: str, max_ending_dist: int = 1) -> List[str]:
    """
    Если word не анализируется, пробуем:
      - найти разбиение на stem + ending'
      - stem должен быть в stems
      - ending' подбираем как ближайшее к хвосту word по Левенштейну среди endings
    """

    w = word
    wl = len(w)

    # ЖЁСТКО: короткие слова не режем на stem+suffix
    if wl <= 4:
        return []

    out = set()

    # перебираем позицию разделения: stem | suffix
    for cut in range(3, wl-1):  # stem>=3, suffix>=2
        stem = w[:cut]
        if stem.lower() not in stems_set:
            continue

        suffix = w[cut:]
        # 1) если suffix точно является окончанием — уже хорошо
        if suffix in endings_set:
            out.add(stem + suffix)
            continue

        # 2) иначе ищем ближайшее окончание
        best = None
        best_d = max_ending_dist + 1
        for e in endings_list:
            # быстрые фильтры по длине
            if abs(len(e) - len(suffix)) > max_ending_dist:
                continue
            d = levenshtein(suffix, e, max_dist=max_ending_dist)
            if d < best_d:
                best_d = d
                best = e
                if best_d == 0:
                    break
        if best is not None and best_d <= max_ending_dist:
            out.add(stem + best)

    return list(out)

def generate_candidates(word: str) -> List[str]:
    cands = set()
    for v in phonetic_variants(word):
        cands.add(v)
        for r in repair_by_endings(v, max_ending_dist=1):
            cands.add(r)
    for r in repair_by_endings(word, max_ending_dist=1):
        cands.add(r)

    # всегда можно оставить исходное
    cands.add(word)
    return list(cands)


import itertools
import math
from collections import Counter

from rapidfuzz import fuzz, process


# ---------------------------------------------------------------------------
# A) Бірнеше фонетикалық ауыстыруды БІР МЕЗГІЛДЕ қолдану
# ---------------------------------------------------------------------------
def phonetic_variants_combo(word: str, phonetic_pairs, max_subs: int = 2) -> List[str]:
    applicable = []
    for a, b in phonetic_pairs:
        if a in word:
            applicable.append((a, b))
        if b in word:
            applicable.append((b, a))

    variants = set()
    for r in range(1, max_subs + 1):
        for combo in itertools.combinations(applicable, r):
            w = word
            for old, new in combo:
                w = w.replace(old, new)
            if w != word:
                variants.add(w)
    return list(variants)


# ---------------------------------------------------------------------------
# B) Бүкіл түбір сөздігі бойынша жуық (fuzzy) іздеу
# ---------------------------------------------------------------------------
def build_fuzzy_index(stems_list_: List[str]):
    return [s.lower() for s in stems_list_]


def fuzzy_stem_candidates(prefix: str, stems_lower_: List[str],
                           limit: int = 5, score_cutoff: int = 72) -> List[str]:
    if len(prefix) < 3:
        return []
    results = process.extract(
        prefix, stems_lower_, scorer=fuzz.ratio,
        limit=limit, score_cutoff=score_cutoff,
    )
    return [r[0] for r in results]


def repair_fuzzy(word: str, stems_lower_: List[str], endings_list_, endings_set_,
                  levenshtein_fn, max_ending_dist: int = 2,
                  stem_score_cutoff: int = 72) -> List[str]:
    w = word
    wl = len(w)
    if wl <= 4:
        return []

    out = set()
    for cut in range(3, wl - 1):
        raw_stem = w[:cut]
        suffix = w[cut:]

        stem_matches = fuzzy_stem_candidates(raw_stem, stems_lower_,
                                              limit=3, score_cutoff=stem_score_cutoff)
        if not stem_matches:
            continue

        for stem in stem_matches:
            if suffix in endings_set_:
                out.add(stem + suffix)
                continue
            best, best_d = None, max_ending_dist + 1
            for e in endings_list_:
                if abs(len(e) - len(suffix)) > max_ending_dist:
                    continue
                d = levenshtein_fn(suffix, e, max_dist=max_ending_dist)
                if d < best_d:
                    best_d, best = d, e
                    if best_d == 0:
                        break
            if best is not None:
                out.add(stem + best)
    return list(out)


def generate_candidates_v2(word: str, phonetic_pairs, stems_lower_,
                            endings_list_, endings_set_, levenshtein_fn) -> List[str]:
    cands = set()
    cands.add(word)

    for v in phonetic_variants_combo(word, phonetic_pairs, max_subs=2):
        cands.add(v)
        for r in repair_fuzzy(v, stems_lower_, endings_list_, endings_set_, levenshtein_fn):
            cands.add(r)

    for r in repair_fuzzy(word, stems_lower_, endings_list_, endings_set_, levenshtein_fn):
        cands.add(r)

    return list(cands)


# ---------------------------------------------------------------------------
# C) Таза лексикалық ranking (BERT/нейрондық LM ЖОҚ)
# ---------------------------------------------------------------------------
def build_stem_frequency(stems_list_: List[str]) -> Counter:
    return Counter(s.lower() for s in stems_list_)


def rank_candidates_lexical(word: str, candidates: List[str], levenshtein_fn,
                             stem_freq_: Counter, stems_set_,
                             cse_analyze_fn) -> List[str]:
    def score(c: str) -> float:
        d = levenshtein_fn(word, c, max_dist=5)
        s = -float(d)
        a = cse_analyze_fn(c)
        if a.ok:
            s += 2.0
        stem_part = a.stem.lower() if a.ok and a.stem else c.lower()
        freq = stem_freq_.get(stem_part, 0)
        if freq > 0:
            s += 0.3 * math.log1p(freq)
        if c.lower() in stems_set_:
            s += 0.5
        return s

    return sorted(candidates, key=score, reverse=True)


# ---------------------------------------------------------------------------
# D) RAW/REF арасындағы "түзетуге келетін" қателердің үлесін өлшеу
# ---------------------------------------------------------------------------
def recoverability_report(raw_words: List[str], ref_words: List[str],
                           levenshtein_fn, max_dist: int = 3) -> dict:
    n = min(len(raw_words), len(ref_words))
    dists = [levenshtein_fn(raw_words[i], ref_words[i], max_dist=99) for i in range(n)]
    recoverable = sum(1 for d in dists if 0 < d <= max_dist)
    unrecoverable = sum(1 for d in dists if d > max_dist)
    exact = sum(1 for d in dists if d == 0)
    return {
        "total_compared": n,
        "already_correct": exact,
        "recoverable_by_mapp": recoverable,
        "needs_better_asr": unrecoverable,
        "recoverable_share_of_errors": recoverable / max(1, recoverable + unrecoverable),
    }


# ---------------------------------------------------------------------------
# Индекстерді бір рет құру + try_fix_word_safe-ті ЕСКІ АТАУМЕН қайта анықтау
# ---------------------------------------------------------------------------


def try_fix_word_safe(word: str, left_ctx: str = "", right_ctx: str = "") -> str:
    w = word.lower()

    w_norm = ASR_NORMALIZE_MAP.get(w, w)
    if w_norm != w:
        return w_norm

    if len(w) <= 3:
        return word

    if w in FUNCTION_WORDS:
        return word

    an0 = cse_analyze_word(w)
    if an0.ok:
        return word

    cands = generate_candidates_v2(w, PHONETIC_PAIRS, stems_lower,
                                    endings_list, endings_set, levenshtein)
    cands = [c for c in cands if c.isalpha() and 2 <= len(c) <= 25]
    if not cands:
        return word

    if levenshtein(w, "бірақ", max_dist=2) <= 1:
        return "бірақ"

    ranked = rank_candidates_lexical(w, cands, levenshtein, stem_freq,
                                      stems_set, cse_analyze_word)
    best = ranked[0]

    if word[:1].isupper():
        best = best[:1].upper() + best[1:]
    return best




def mapp_asr_process(text: str) -> str:
    tokens = tokenize_keep_punct(text)
    out = []

    # для контекста берём соседние токены-слова
    word_positions = [i for i,t in enumerate(tokens) if is_word(t)]
    word_set = set(word_positions)

    for i, tok in enumerate(tokens):
        tlow = normalize(tok)

        # стоп-слова как есть
        if tlow in stop_words:
            out.append(tok)
            continue

        # не слово (пунктуация/числа) — как есть
        if not is_word(tok):
            out.append(tok)
            continue

        # если анализируется — оставляем
        an = cse_analyze_word(tlow)
        if an.ok:
            out.append(tok)
            continue

        # иначе: генерация кандидатов + выбор
        cands = generate_candidates(tlow)

        # простой контекст: ближайшее слово слева/справа
        left = ""
        right = ""
        # слева
        j = i-1
        while j >= 0:
            if j in word_set:
                left = normalize(tokens[j])
                break
            j -= 1
        # справа
        j = i+1
        while j < len(tokens):
            if j in word_set:
                right = normalize(tokens[j])
                break
            j += 1

        #best = pick_best_candidate(tlow, cands, left, right)
        best = try_fix_word_safe(tlow, left, right)

        # сохраняем исходный регистр (минимально): если токен был с заглавной
        if tok[:1].isupper():
            best = best[:1].upper() + best[1:]
        out.append(best)

    # аккуратно склеим: чтобы не было пробелов перед пунктуацией
    # 1) сначала join с пробелами
    joined = " ".join(out)
    # 2) уберём пробелы перед знаками
    joined = re.sub(r"\s+([.,!?;:%)\]\}»])", r"\1", joined)
    joined = re.sub(r"([(\[\{«])\s+", r"\1", joined)
    return joined


In [4]:
load_resources(STEMS_PATH, ENDINGS_PATH, STOP_PATH)
import time
t0 = time.time(); print(mapp_asr_process("Биз бирак казир  оқимаймыз, себеби ауа райы жаман."), round(time.time()-t0,1), "с")

stems: 103615 endings: 5994 stopwords: 188
Біз бірақ қазір оқимаймыз, себебі ауа райы жаман. 0.0 с


In [5]:
from tqdm.auto import tqdm

def collapse_loops(text, max_run=2):
    out, prev, run = [], None, 0
    for w in str(text).split():
        if w == prev:
            run += 1
            if run <= max_run: out.append(w)
        else:
            prev, run = w, 1; out.append(w)
    return " ".join(out)

JOBS = [
  ("Whisper large-v3",
   D+"/mapp_v2_batch_results_asr_upgrade.csv", False,
   D+"/largev3_lex/mapp_v2_batch_results_largev3_lex.csv"),
]
for name, src, do_loop, dst in JOBS:
    d = pd.read_csv(src)
    raw = d["raw"].astype(str).apply(collapse_loops) if do_loop else d["raw"].astype(str)
    fixed = [mapp_asr_process(t) for t in tqdm(raw, desc=name)]
    o = pd.DataFrame({"duration": d["duration"], "stem": d["stem"], "ref": d["ref"], "raw": raw, "fixed": fixed})
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    o.to_csv(dst, index=False)
    ch = (o.raw != o.fixed).sum()
    print(name, "| файл:", len(o), "| fixed != raw болған файл:", ch, "| сақталды:", dst)
    assert ch > 0, "fixed әлі raw-ға тең — лексикалық түзеткіш жұмыс істемеді"

Whisper large-v3:   0%|          | 0/100 [00:00<?, ?it/s]

Whisper large-v3 | файл: 100 | fixed != raw болған файл: 97 | сақталды: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/largev3_lex/mapp_v2_batch_results_largev3_lex.csv
